# Perception SLM — Image v0 training on Kaggle (free T4)

Runs the **real** Phase-2 pipeline on GPU: build COCO shards → SigLIP-init alignment → LoRA instruction tuning → quantize/bench → push to the HuggingFace Hub.

**Before running:** in the Kaggle notebook sidebar set **Accelerator = GPU T4 x2** (or P100) and **Internet = On**. Add your HF token under **Add-ons → Secrets** as `HF_TOKEN` (write scope). Optional: add `WANDB_API_KEY` for live metrics.

Each stage checkpoints, so if the 12h session ends you re-run and resume.

## 1 · Clone the repo & install deps
Torch+CUDA is preinstalled on Kaggle; we only add the rest. Set `REPO_URL` to your GitHub repo.

In [ ]:
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"  # <-- EDIT ME

import os
os.chdir('/kaggle/working')
if not os.path.exists('repo'):
    !git clone $REPO_URL repo
os.chdir('/kaggle/working/repo')
!git pull --ff-only || true
print('cwd:', os.getcwd())

In [ ]:
# Install project deps (skip torch/torchvision — Kaggle already has the CUDA build).
!pip -q install transformers datasets webdataset peft hydra-core omegaconf einops \
    huggingface_hub wandb pyarrow onnx onnxruntime
!pip -q install -e .  # makes `common`, `image_model`, `data_pipeline`, ... importable

In [ ]:
import torch, sys
sys.path.insert(0, '/kaggle/working/repo/src')
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Enable GPU in the notebook settings!'

## 2 · Log in to HuggingFace (+ optional W&B)

In [ ]:
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
os.environ['HF_TOKEN'] = sec.get_secret('HF_TOKEN')
from huggingface_hub import login
login(os.environ['HF_TOKEN'])

try:
    os.environ['WANDB_API_KEY'] = sec.get_secret('WANDB_API_KEY')
    import wandb; wandb.login(); WANDB = 'online'
except Exception as e:
    print('W&B off (no key):', e); WANDB = 'disabled'

## 3 · Build real COCO shards (Week 3)
Streams from `clip-benchmark/wds_mscoco_captions`, cleans, splits, shards. Bump `--limit` for more data.

In [ ]:
!python -m data_pipeline.build --dataset coco --limit 3000 \
    --out data/processed/coco_train --maxcount 1000
!cat data/processed/coco_train/DATA_CARD.md

## 4 · Stage-2 alignment on GPU (Week 5)
Frozen SigLIP + resampler + from-scratch decoder; trains the connector. Held-out eval every N steps, saves `best.pt`.

In [ ]:
from common.config import load_config
from training.image_v0 import run_image_v0

cfg = load_config('image_v0_gpu')
cfg.data.shards_dir = 'data/processed/coco_train'
cfg.tracking.mode = WANDB
cfg.optim.steps = 2000          # raise as your GPU budget allows
cfg.data.batch_size = 32
out = run_image_v0(cfg)
out

In [ ]:
# Inspect held-out captions from the best checkpoint.
from omegaconf import OmegaConf
from common.checkpoint import load_checkpoint
from common.tokenizer import TinyTokenizer
from image_model.model import ImageVLM
from training.image_v0 import _load_split
from eval.evaluate import evaluate

tok = TinyTokenizer()
m = OmegaConf.to_container(cfg, resolve=True); m['decoder']['vocab_size'] = tok.vocab_size
model = ImageVLM.from_config(m).to('cuda')
load_checkpoint(out['best_checkpoint'] or out['checkpoint'], model, map_location='cuda')
val = _load_split(cfg, 'val', cfg.seed)
rep = evaluate(model, val, batch_size=32, device='cuda')   # pass device explicitly
print(rep.as_dict()); rep.samples[:4]

## 5 · Stage-3 LoRA instruction tuning (Week 6)
Trains connector + decoder LoRA and reports held-out VQA vs. the majority baseline.

> Note: this uses the **synthetic VQA** task (real, measurable, offline). Swapping in real VQAv2/LLaVA-Instruct streams through the same `data_pipeline` + chat template — that's the next extension.

In [ ]:
# Standalone tiny-VQA instruction demo (its own tiny config; not the SigLIP model).
# Warm-starting from the SigLIP alignment ckpt is skipped: different architecture.
# Unifying them (real VQA data at 224px on the SigLIP model) is the next extension.
from training.instruct import run_instruct
icfg = load_config('instruct')
icfg.tracking.mode = WANDB
icfg.optim.steps = 800
icfg.init_from = None
ires = run_instruct(icfg)
print('VQA', ires['vqa_before'], '->', ires['vqa_after'], '| baseline', ires['baseline_accuracy'],
      '| beats', ires['beats_baseline'])
ires['samples']

## 6 · Quantize + benchmark (Week 7)

In [ ]:
from serving.offline.export import export_quantized
from serving.offline.bench import benchmark

cpu_model = model.to('cpu')
print('bench:', benchmark(cpu_model, image_size=cfg.image.image_size, batch=8).as_dict())
print('quant:', export_quantized(cpu_model, 'outputs/image_v0_gpu/model_int8.pt'))

## 7 · Deploy to the HuggingFace Hub
Pushes the checkpoint, config, int8 weights, and an auto model card.

In [ ]:
REPO_ID = 'LNTTushar/perception-slm-image-v0'  # <-- your HF repo
!python scripts/push_to_hf.py \
    --repo-id $REPO_ID \
    --checkpoint {out['best_checkpoint'] or out['checkpoint']} \
    --config image_v0_gpu \
    --int8 outputs/image_v0_gpu/model_int8.pt

## Next
- Raise `optim.steps` / `--limit` for a stronger model (checkpoint + resume across sessions).
- Download `best.pt` to your laptop and run `python scripts/ask_image.py ... --quantize` offline.
- Extend `data_pipeline` with real VQAv2/LLaVA-Instruct for real instruction tuning.